In [ ]:
# Lab type: debug
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: ML-Based Forecasting — Features from Time
# Task: The feature-engineering pipeline below contains 3 bugs. Each runs without
#       error and each makes the evaluation score BETTER than the truth. Find each
#       bug, explain it in the markdown cell below it, and fix it in the fix cell.

# Lab: Debugging a Leaky Feature Pipeline

A one-day-ahead orders model, built the way AI assistants typically build it.
It reports excellent accuracy. All three bugs are from the course's leakage
catalogue — and all three inflate the score.

A **reference honest evaluation** is provided first so every bug's inflation is
measurable.

**Outputs are cleared.** Run each cell to generate results.

## Setup and honest reference

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Reference — honest one-day-ahead evaluation (correct features, chronological split)
df = pd.DataFrame({"orders": orders})
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["lag_1"] = df["orders"].shift(1)
df["lag_7"] = df["orders"].shift(7)
df["roll_7"] = df["orders"].shift(1).rolling(7).mean()
df = df.dropna()

FEATS = ["dayofweek", "month", "lag_1", "lag_7", "roll_7"]
train, test = df.iloc[:-90], df.iloc[-90:]

ref = HistGradientBoostingRegressor(random_state=0).fit(train[FEATS], train["orders"])
mae_ref = mean_absolute_error(test["orders"], ref.predict(test[FEATS]))
print(f"MAE (honest): {mae_ref:.1f}")

## Bug 1: The rolling window

A teammate 'simplified' the rolling feature. The model gets better. Did it?

In [ ]:
# --- BUGGY CODE (Bug 1) ---
# Review this feature — what window does row t actually see?
df_b1 = df.copy()
df_b1["roll_7"] = df_b1["orders"].rolling(7).mean()      # ← 'simplified'
df_b1 = df_b1.dropna()

tr, te = df_b1.iloc[:-90], df_b1.iloc[-90:]
m1 = HistGradientBoostingRegressor(random_state=0).fit(tr[FEATS], tr["orders"])
mae_b1 = mean_absolute_error(te["orders"], m1.predict(te[FEATS]))
print(f"MAE (bug 1):  {mae_b1:.1f}")
print(f"MAE (honest): {mae_ref:.1f}   inflation: {mae_ref - mae_b1:+.1f}")

**Explain the bug:** For the row whose target is day *t*, which days does this
`roll_7` average? Which single value inside that window makes the feature illegal,
and why can the deployed model never compute this feature?

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**The bug:** Without `shift(1)`, `rolling(7).mean()` at row *t* averages days
*t−6 … t* — including day *t* itself, which is the row's own label. One-seventh of the
answer is inside the feature.

**Why deployment breaks:** At prediction time, day *t*'s orders don't exist yet — the
production pipeline would silently compute the window *t−6 … t−1* plus a missing value
(or shift to yesterday's window), a *different feature* from the one the model trained
on. Training metrics inflate; live accuracy reverts.

**Correct approach:** `shift(1).rolling(7).mean()` — window *t−7 … t−1*, computable at
prediction time, identical in training and deployment.

</details>

In [ ]:
# Fix for Bug 1: shift before rolling
df_f1 = df.copy()
df_f1["roll_7"] = df_f1["orders"].shift(1).rolling(7).mean()
df_f1 = df_f1.dropna()

tr, te = df_f1.iloc[:-90], df_f1.iloc[-90:]
f1 = HistGradientBoostingRegressor(random_state=0).fit(tr[FEATS], tr["orders"])
print(f"MAE (fix 1): {mean_absolute_error(te['orders'], f1.predict(te[FEATS])):.1f}  (should match honest)")

## Bug 2: Filling the warm-up NaNs

The first 7 rows of lag/rolling features are NaN. The pipeline 'cleans' them.

In [ ]:
# --- BUGGY CODE (Bug 2) ---
# Review this NaN handling — where do the filled values come from?
df_b2 = pd.DataFrame({"orders": orders})
df_b2["dayofweek"] = df_b2.index.dayofweek
df_b2["month"] = df_b2.index.month
df_b2["lag_1"] = df_b2["orders"].shift(1)
df_b2["lag_7"] = df_b2["orders"].shift(7)
df_b2["roll_7"] = df_b2["orders"].shift(1).rolling(7).mean()
df_b2 = df_b2.bfill()          # ← 'clean up the NaNs'

print(df_b2.head(8)[["orders", "lag_1", "lag_7", "roll_7"]])

**Explain the bug:** Look at the printed head. For the first row, where did `lag_7`'s
value come from — which actual day? Why is backward-fill *specifically* wrong for
temporal features, when it's often fine for cross-sectional data? How should warm-up
rows be handled instead?

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**The bug:** `bfill()` fills each NaN with the next valid value *below* it — i.e. from a
*later date*. Row 1's `lag_7` receives the first computed lag_7 value, which belongs to
day 8 and encodes day 1's orders... observed *after* the feature row it now sits in.
Every filled cell contains information from the future of its row.

**Why it matters here specifically:** In cross-sectional data, row order is arbitrary,
so bfill is just borrowing from a neighbour. In a time-indexed frame, "the next row" is
"the future" — bfill is temporal leakage by construction.

**Correct approach:** Drop the warm-up rows (`dropna()`), as the honest reference does.
Seven rows out of 1,096 cost nothing; a leak costs trust in every metric.

</details>

In [ ]:
# Fix for Bug 2: drop warm-up rows instead of filling from the future
df_f2 = pd.DataFrame({"orders": orders})
df_f2["dayofweek"] = df_f2.index.dayofweek
df_f2["month"] = df_f2.index.month
df_f2["lag_1"] = df_f2["orders"].shift(1)
df_f2["lag_7"] = df_f2["orders"].shift(7)
df_f2["roll_7"] = df_f2["orders"].shift(1).rolling(7).mean()
df_f2 = df_f2.dropna()
print(f"rows: {len(df_f2)} (7 warm-up rows dropped, none fabricated)")

## Bug 3: The scaler

For a linear variant of the model, the pipeline standardises the features.

In [ ]:
# --- BUGGY CODE (Bug 3) ---
# Review this preprocessing — what does the scaler learn, and from which rows?
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

X, y = df[FEATS], df["orders"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)               # ← fit on all 3 years

X_tr, X_te = X_scaled[:-90], X_scaled[-90:]
y_tr, y_te = y.iloc[:-90], y.iloc[-90:]

ridge_bug = Ridge().fit(X_tr, y_tr)
print(f"MAE (bug 3): {mean_absolute_error(y_te, ridge_bug.predict(X_te)):.1f}")
print(f"scaler learned lag_1 mean = {scaler.mean_[2]:.1f} from ALL rows "
      f"(train-only mean would be {X['lag_1'].iloc[:-90].mean():.1f})")

**Explain the bug:** The split is chronological — so where is the leak? What did the
scaler learn from the last 90 days, and why does that matter *more* on a trending
series than on a stationary one?

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**The bug:** `fit_transform` on the full matrix computes feature means/variances using
the 90 test days — fit-on-all leakage across the time boundary, even though the split
itself is chronological.

**Why trend makes it worse:** On a trending series, the test period sits at the highest
level the series ever reaches, so the full-data mean is pulled noticeably above the
train-only mean (the printout shows the gap). The training features arrive pre-centred
around information about *where the future ends up* — a hint no deployed model gets.

**Correct approach:** Fit the scaler on training rows only (`fit` on train,
`transform` on test) — or make it structural with
`Pipeline([("scaler", StandardScaler()), ("model", Ridge())])` fit on the training
window, as Lesson 7 does inside every backtest fold.

</details>

In [ ]:
# Fix for Bug 3: scaler inside a Pipeline, fit on train only
from sklearn.pipeline import Pipeline

pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge())])
pipe.fit(X.iloc[:-90], y.iloc[:-90])
print(f"MAE (fix 3): {mean_absolute_error(y.iloc[-90:], pipe.predict(X.iloc[-90:])):.1f}")

## Summary

> **For each bug, complete the sentence in one line.**

1. **Rolling:** `rolling(7)` without `shift(1)` puts ________ inside its own feature window.
2. **bfill:** Backward-fill is temporal leakage because in a time-indexed frame, the next row is ________.
3. **Scaler:** Even with a chronological split, `fit_transform` on the full matrix leaks because ________.

<details>
<summary>🔑 Reveal summary answers</summary>

1. ...puts **the row's own label (day t's orders)** inside its own feature window.
2. ...the next row is **the future**.
3. ...because **the scaler's means/variances are computed from the test period — on a
   trending series they encode where the future ends up**.

</details>